Consumer: Someone waiting on the other end of the producer pipe
Is processing and doing something (e.g., putting these events in a datalake)
In our case, it's to print and show them.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

In [ ]:
# This is from the consumer.py file but is not in the example during setup. 
# We need to add the parent directory of this notebook to the Python path so we can import our models module. 
# This allows us to use the Ride dataclass and the deserializer function we defined in models.py.
sys.path.insert(0, str(Path().parent.parent))

In [15]:
# Connect to Kafka as a consumer. auto_offset_reset='earliest' means we start reading from the beginning of the topic 
# (without this, new consumers default to latest and only see new messages). 
# group_id identifies this consumer group - Kafka tracks how far each group has read, so restarting with the same group ID continues where it left off:

from kafka import KafkaConsumer

# Define the Kafka server and topic we want to consume from. 
# In a real setup, the server would be the address of your Kafka cluster, but for local testing, we can use localhost.
server = 'localhost:9092'
topic_name = 'rides'

# Need the serializer that turns data from the sequence of bytes that Kafka sends us, into a Python object that we can work with.
consumer = KafkaConsumer(
    topic_name, # Kafka topic to subscribe to
    bootstrap_servers=[server], # Kafka server address
    auto_offset_reset='earliest', # start from the earliest message if no offset is committed for this group
    group_id='rides-console', # consumer group ID for tracking offsets
    value_deserializer=ride_deserializer # This is the function we defined in models.py to convert bytes back into a Ride object.
)

In [16]:
# Read messages and print them. 
# Since value_deserializer returns a Ride, message.value is already a Ride object - no extra conversion needed:

from datetime import datetime

print(f"Listening to {topic_name}...")

# For demo purposes, we'll just read and print 10 messages, then stop. In a real consumer, you'd typically run indefinitely.
count = 0
for message in consumer:
    ride = message.value # message.value is already a Ride object thanks to our deserializer
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000) # Convert milliseconds to seconds for datetime
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, " # print some key details from the ride
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1 # Increment our message count
    if count >= 10: # After 10 messages, break the loop for demo purposes. In a real consumer, you'd likely want to keep running indefinitely.
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...


NameError: name 'Ride' is not defined